# Practical 1: End-to-End Time Series Analysis & Forecasting
## Dataset: AirPassengers (Monthly International Airline Passengers 1949 - 1960)

---
### 📘 Beginner's Guide: What is Time Series Analysis?
A **Time Series** is a sequence of data points recorded at regular time intervals (e.g., daily, monthly, yearly). Unlike standard statistical data where observations are independent, time series data has a chronological order where past values influence future values.

**The 4 Key Components of Time Series Data:**
1. **Level**: The baseline value of the series.
2. **Trend**: The long-term upward or downward movement over time.
3. **Seasonality**: Repeating patterns or cycles that occur at fixed time intervals (e.g., higher airline travel every summer).
4. **Residual / Noise**: Random, unpredictable fluctuations left after removing trend and seasonality.

---

### Step 1: Import Python Libraries
**Why we write this code:**
- `pandas` & `numpy`: Essential for data loading, index manipulation, and array calculations.
- `matplotlib.pyplot` & `seaborn`: For plotting graphs and visual diagnostic analysis.
- `statsmodels.tsa`: Contains specialized time series functions for decomposition, forecasting models (SES, Holt, Holt-Winters), and statistical stationarity tests (ADF, KPSS).
- `pymannkendall`: Performs statistical hypothesis testing to confirm if a monotonic trend exists.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.seasonal import seasonal_decompose

### Step 2: Load the Dataset
**Why we write this code:**
- `pd.read_csv("AirPassengers.xls")`: Reads the spreadsheet file into a Pandas DataFrame.
- `data.head(5)`: Displays the first 5 rows to inspect column names (`Month` and `#Passengers`).

In [ ]:
data = pd.read_csv("AirPassengers.xls")
data.head(5)

### Step 3: Inspect Dataset Dimensions
**Why we write this code:**
- `data.shape`: Returns `(144, 2)`, meaning we have 144 monthly observations (12 years × 12 months) and 2 columns.

In [ ]:
data.shape

### Step 4: Datetime Formatting and Indexing (Crucial Step!)
**Why we write this code:**
- `pd.to_datetime(data["Month"])`: Converts plain text strings (like `"1949-01"`) into official Pandas `Timestamp` objects.
- `data.set_index("Month")`: Sets the `Month` column as the DataFrame index. In time series modeling, models require dates as index labels so they can infer time frequency (Monthly).

In [ ]:
# convert column year in date time format (truncating to month)
data["Month"] = pd.to_datetime(data["Month"])
data = data.set_index("Month")

In [ ]:
data.head(5)

In [ ]:
data.shape

### Step 5: Visualize the Raw Time Series Data
**Why we write this code:**
Before running any complex math, we must ALWAYS plot the raw series to understand its behavior visually.

**📊 Detailed Graph Explanation (Graph 1 - Line Plot):**
- **X-Axis**: Represents time (Years 1949 to 1960).
- **Y-Axis**: Total number of international airline passengers in thousands.
- **Observation**: Notice how the blue curve goes up steadily over time (**Upward Trend**). Furthermore, notice how every summer (July/August), passenger counts spike upward, and the height of these seasonal spikes expands as passenger counts grow (**Expanding Seasonality**).

In [ ]:
sns.lineplot(data)
plt.ylabel("#Passengers")
plt.title("AirPassengers Monthly Raw Time Series (1949 - 1960)")
plt.show()

**Key Takeaway from Visual Inspection:**
We can see that `#Passengers` is increasing over time with strong seasonality. Because the seasonal oscillation height expands as the overall trend increases, this series follows a **Multiplicative Model**.

### Step 6: Seasonal Decomposition (Multiplicative Model)
**Why we write this code:**
- `seasonal_decompose(..., model="multiplicative", period=12)`: Mathematically splits raw data into 4 component sub-plots.

**📊 Detailed Graph Explanation (Graph 2 - Decomposition Grid):**
1. **Observed (Top Panel)**: The raw original data showing trend + seasonality + noise.
2. **Trend (Second Panel)**: The smooth underlying upward growth line, ignoring monthly ups and downs.
3. **Seasonal (Third Panel)**: The exact repeating 12-month annual wave pattern (showing peaks every July and dips every November).
4. **Resid (Bottom Panel)**: Residual random noise left over after subtracting trend and seasonality.

In [ ]:
# decomposition of the time series - multiplicative model
result = seasonal_decompose(data[["#Passengers"]], model = "multiplicative", period = 12)
result.plot()
plt.show()

### Step 7: Mann-Kendall Statistical Test for Monotonic Trend
**Why we write this code:**
Rather than just guessing there is a trend from visual inspection, we perform a formal non-parametric statistical hypothesis test.

**Statistical Rules:**
- **$H_0$ (Null Hypothesis)**: There is NO monotonic trend in the series.
- **$H_1$ (Alternative Hypothesis)**: A monotonic trend exists.
- **Decision Rule**: If `p-value < 0.05`, we reject $H_0$ and statistically prove that a trend exists.

In [ ]:
# Perform the Mann-Kendall test
import pymannkendall as mk
mk.original_test(data["#Passengers"])

### Step 8: Sequential Train-Test Split (70% Train / 30% Test)
**Why we write this code:**
- `data[:int(data.shape[0]*0.7)]`: Takes the first 70% (~100 months) for training our models.
- `data[int(data.shape[0]*0.7):]`: Keeps the remaining 30% (~44 months) as the test set to evaluate forecast accuracy.
- **CRITICAL RULE**: Time series data MUST be split chronologically in order. NEVER shuffle time series data because future predictions rely on historical sequence!

In [ ]:
train_df = data[:int(data.shape[0]*0.7)]
test_df = data[int(data.shape[0]*0.7):]
train_df.head(5)

### Step 9: Model 1 — Single Exponential Smoothing (SES)
**Why we write this code:**
- `SimpleExpSmoothing(train_df)`: Fits the simplest exponential smoothing model, which estimates only a baseline Level parameter ($alpha$).
- `model_single_fit.forecast(len(test_df))`: Generates predictions for the next 44 months.

**📊 Detailed Graph Explanation (Graph 3 - SES Forecast Plot):**
- **Original Data (Blue)**: Actual historical values.
- **Fitted Values (Orange)**: In-sample model fit.
- **Forecast (Green)**: A completely **flat horizontal line**.
- **Why is it flat?** Because SES does NOT include trend ($eta$) or seasonal ($gamma$) parameters. Therefore, its multi-step forecast is always a constant horizontal line equal to the last estimated level.

In [ ]:
# single exponential smoothing model
from statsmodels.tsa.api import SimpleExpSmoothing
model = SimpleExpSmoothing(train_df)
model_single_fit = model.fit()
forecast_single = model_single_fit.forecast(len(test_df))
print(forecast_single)

In [ ]:
model_single_fit.params

In [ ]:
plt.plot(data, label ="Original Data")
plt.plot(model_single_fit.fittedvalues, label= "Fitted values")
plt.plot(forecast_single, label = "Forecast")
plt.xlabel("Year")
plt.ylabel("#Passengers")
plt.title("Single Exponential Smoothing (Flat Forecast)")
plt.legend()
plt.show()

### Step 10: Model 2 — Double Exponential Smoothing (Holt's Linear Model)
**Why we write this code:**
- `Holt(train_df)`: Fits level ($alpha$) and trend ($eta$) parameters.

**📊 Detailed Graph Explanation (Graph 4 - Holt's Model Forecast Plot):**
- **Forecast Line (Green)**: A **straight upward-sloping line** into the test period.
- **Why is it a sloped straight line?** Holt's model captures the linear upward trend slope ($eta$), but because it lacks seasonal parameters ($gamma$), it cannot capture seasonal peaks and troughs.

In [ ]:
# DOUBLE EXPONENTIAL SMOOTHING (Holt's model)
from statsmodels.tsa.api import Holt
model_double = Holt(train_df)
model_double_fit = model_double.fit()
forecast_double = model_double_fit.forecast(len(test_df))
print(forecast_double)

In [ ]:
model_double_fit.params

In [ ]:
plt.plot(data, label ="Original Data")
plt.plot(model_double_fit.fittedvalues, label= "Fitted values")
plt.plot(forecast_double, label = "Forecast")
plt.xlabel("Year")
plt.ylabel("#Passengers")
plt.title("Double Exponential Smoothing (Holt's Model)")
plt.legend()
plt.show()

### Step 11: Model 3 — Triple Exponential Smoothing (Holt-Winters Additive)
**Why we write this code:**
- `ExponentialSmoothing(..., trend="add", seasonal="add", seasonal_periods=12)`: Fits level ($alpha$), trend ($eta$), and additive seasonal components ($gamma$).

**📊 Detailed Graph Explanation (Graph 5 - Holt-Winters Additive Plot):**
- **Forecast Line (Green)**: Shows repeating seasonal waves.
- **Why does it underestimate late peaks?** Additive seasonality assumes seasonal fluctuations add a fixed constant number of passengers (e.g. +50 passengers every July) regardless of how high the overall trend rises.

In [ ]:
# TRIPLE EXPONENTIAL SMOOTHING (HOLT_WINTER'S MODEL) - Additive
from statsmodels.tsa.api import ExponentialSmoothing
model_triple = ExponentialSmoothing(train_df, seasonal_periods = 12, trend = "add", seasonal = "add")
model_triple_fit = model_triple.fit()
print(model_triple_fit.params)
forecast_triple = model_triple_fit.forecast(len(test_df))
print(forecast_triple)

In [ ]:
plt.plot(data, label ="Original Data")
plt.plot(model_triple_fit.fittedvalues, label= "Fitted values")
plt.plot(forecast_triple, label = "Forecast")
plt.xlabel("Year")
plt.ylabel("#Passengers")
plt.title("Triple Exponential Smoothing (Holt-Winters Additive)")
plt.legend()
plt.show()

### Step 12: Model 4 — Triple Exponential Smoothing (Holt-Winters Multiplicative)
**Why we write this code:**
- `ExponentialSmoothing(..., trend="add", seasonal="mul", seasonal_periods=12)`: Fits level ($alpha$), trend ($eta$), and multiplicative seasonal components ($gamma$).

**📊 Detailed Graph Explanation (Graph 6 - Holt-Winters Multiplicative Plot):**
- **Forecast Line (Green)**: Shows **expanding seasonal waves** that grow taller as the trend rises.
- **Why is this the best model?** In AirPassengers, seasonal peaks grow proportionally with total passengers. Multiplicative seasonality models percentage growth rather than fixed numbers, matching actual passenger counts almost perfectly.

In [ ]:
# TRIPLE EXPONENTIAL SMOOTHING (HOLT_WINTER'S MODEL) - Multiplicative
model_triple_mul = ExponentialSmoothing(train_df, seasonal_periods = 12, trend = "add", seasonal = "mul")
model_triple_fit_mul = model_triple_mul.fit()
print(model_triple_fit_mul.params)
forecast_triple_mul = model_triple_fit_mul.forecast(len(test_df))
print(forecast_triple_mul)

In [ ]:
plt.plot(data, label ="Original Data")
plt.plot(model_triple_fit_mul.fittedvalues, label= "Fitted values")
plt.plot(forecast_triple_mul, label = "Forecast")
plt.xlabel("Year")
plt.ylabel("#Passengers")
plt.title("Triple Exponential Smoothing (Holt-Winters Multiplicative)")
plt.legend()
plt.show()

### Step 13: Evaluate Forecast Accuracy (MAPE)
**Why we write this code:**
- `mean_absolute_percentage_error(y_true, y_pred)`: Computes Mean Absolute Percentage Error (MAPE) between actual test values and predicted values. Lower percentage = higher accuracy.
- **Result**: Multiplicative model achieves a significantly lower MAPE (~2.4% error vs 5.3% for Additive).

In [ ]:
# testing for the accuracy of the two models
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
mape_test_add = mean_absolute_percentage_error(test_df["#Passengers"], forecast_triple)
print("MAPE Test for Additive Model:", mape_test_add)
mape_test_mul = mean_absolute_percentage_error(test_df["#Passengers"], forecast_triple_mul)
print("MAPE Test for Multiplicative Model:", mape_test_mul)

---
## 🔄 Part B: Stationarity Testing & Multi-Stage Differencing Pipeline

### What is Stationarity and Why is it Essential?
A time series is **Stationary** if its statistical properties (mean, variance, autocorrelation) remain constant over time.
- Most advanced time series models (like ARIMA / SARIMA) require data to be stationary before fitting.
- If data has trend or seasonality, it is non-stationary and must be differenced.

**The Dual Hypothesis Testing Approach:**
1. **ADF (Augmented Dickey-Fuller) Test**:
   - $H_0$: Series is Non-Stationary (Has a unit root).
   - $H_1$: Series is Stationary.
   - Rule: **$p < 0.05 ightarrow$ Stationary**.
2. **KPSS Test**:
   - $H_0$: Series is Trend-Stationary.
   - $H_1$: Series is Non-Stationary.
   - Rule: **$p > 0.05 ightarrow$ Stationary**.
3. **Golden Rule**: Series is FULLY stationary only when **both** conditions pass ($p_{	ext{ADF}} < 0.05$ AND $p_{	ext{KPSS}} > 0.05$).

### Step 14: Stationarity Test 1 — Raw Data
**Why we write this code:**
Test raw un-differenced data to confirm non-stationarity.

In [ ]:
# ADF Test
# H0: Series is not stationary, i.e., series has a unit root
# H1: Series is stationary, i.e., series has no unit root
from statsmodels.tsa.stattools import adfuller
result = adfuller(data["#Passengers"])
print("ADF Statistic:", result[0])
print("p-value:", result[1])
# p-value > 0.05: fail to reject H0, i.e., series is not stationary

In [ ]:
# KPSS Test
# H0: Series is trend stationary, i.e., series has no unit root
# H1: Series is non-stationary, i.e., series has a unit root
from statsmodels.tsa.stattools import kpss
kp = kpss(data["#Passengers"])
p = kp[1]
print("p-value for KPSS Test(untransformed) =", p)
# p-value = 0.01 < 0.05: Reject H0, i.e., series is non stationary
# Here, since both conditions are not satisfied, we conclude that the series is NOT stationary.

### Step 15: Raw Data ACF and PACF Plots
**Why we write this code:**
- `plot_acf()`: Plots Autocorrelation Function.
- `plot_pacf()`: Plots Partial Autocorrelation Function.

**📊 Detailed Graph Explanation (Graphs 7 & 8 - Raw ACF & PACF):**
- **ACF Plot**: Shows very slow, linear decay across 40 lags with wave-like bumps every 12 lags. Slow decay confirms strong non-stationarity and memory.
- **PACF Plot**: Shows a dominant single spike at Lag 1.

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plt.figure(figsize = (14, 4))
plot_acf(data["#Passengers"])
plt.show()
plot_pacf(data["#Passengers"])
plt.show()

### Step 16: Stage 1 Differencing — First Non-Seasonal Differencing `diff()`
**Why we write this code:**
- `data["#Passengers"].diff().dropna()`: Subtracts previous month from current month ($Y_t - Y_{t-1}$). Removes linear trend.

**📊 Detailed Graph Explanation (Graphs 9, 10 & 11 - Non-Seasonal Diff Plots):**
- **Line Plot**: Centers around 0 (trend is removed).
- **ACF Plot**: Shows large recurring spikes at seasonal lags 12, 24, 36.
- **Conclusion**: Trend is gone, but strong **seasonality remains**.

In [ ]:
# NON SEASONAL DIFFERENCING
diff = data["#Passengers"].diff().dropna()
plt.figure(figsize = (14, 3))
plt.grid()
plt.plot(diff)
plt.title("Non-Seasonal Differenced Series")
plt.show()

In [ ]:
# ADF Test on Non-Seasonal Differenced Series
result = adfuller(diff)
print("ADF Statistic:", result[0])
print("p-value:", result[1])
# Since p-value < 0.05, reject H0, therefore series is stationary according to ADF.

In [ ]:
# KPSS Test on Non-Seasonal Differenced Series
kp = kpss(diff)
print("p-value for KPSS Test =", kp[1])
# p-value < 0.05, reject H0, i.e., series is still non-stationary according to KPSS.

In [ ]:
plot_acf(diff)
plt.show()
plot_pacf(diff)
plt.show()
# According to the plots, seasonality is still present.

### Step 17: Stage 2 Differencing — Seasonal Differencing `diff(12)`
**Why we write this code:**
- `data["#Passengers"].diff(periods=12).dropna()`: Subtracts same month from previous year ($Y_t - Y_{t-12}$). Removes seasonality.

**📊 Detailed Graph Explanation (Graphs 12, 13 & 14 - Seasonal Diff Plots):**
- **Line Plot**: Seasonal peaks are removed.
- **ACF Plot**: Shows gradual decay across initial lags.
- **Conclusion**: Seasonality is gone, but **residual trend/drift remains**.

In [ ]:
# SEASONAL DIFFERENCING
sdiff = data["#Passengers"].diff(periods = 12).dropna()
plt.figure(figsize = (14, 3))
plt.grid()
plt.plot(sdiff)
plt.title("Seasonal Differenced Series (period=12)")
plt.show()

In [ ]:
# ADF Test on Seasonal Differenced Series
result = adfuller(sdiff)
print("ADF Statistic:", result[0])
print("p-value:", result[1])

In [ ]:
# KPSS Test on Seasonal Differenced Series
kp = kpss(sdiff)
print("p-value for KPSS Test =", kp[1])

In [ ]:
plot_acf(sdiff)
plt.show()
plot_pacf(sdiff)
plt.show()

### Step 18: Stage 3 Differencing — Combined Seasonal & Non-Seasonal Differencing `sdiff.diff()`
**Why we write this code:**
- `sddiff = sdiff.diff().dropna()`: Applies non-seasonal differencing on the seasonal differenced series. Removes BOTH trend and seasonality.

**📊 Detailed Graph Explanation (Graphs 15, 16 & 17 - Combined Diff Plots):**
- **Line Plot**: Fluctuates randomly around 0 with constant variance.
- **ACF & PACF Plots**: Lag 0 is 1.0; all subsequent lags immediately drop inside the blue shaded 95% confidence bounds.
- **Statistical Tests**: Passes ADF ($p < 0.05$) AND passes KPSS ($p > 0.05$).
- **Final Conclusion**: The series is now **FULL STATIONARY** and ready for Box-Jenkins SARIMA modeling!

In [ ]:
# SEASONAL AND NON SEASONAL DIFFERENCING
sddiff = sdiff.diff().dropna()
plt.figure(figsize = (14, 3))
plt.grid()
plt.plot(sddiff)
plt.title("Combined Seasonal & Non-Seasonal Differenced Series")
plt.show()

In [ ]:
# ADF Test on Combined Differenced Series
result = adfuller(sddiff)
print("ADF Statistic:", result[0])
print("p-value:", result[1])
# Series is stationary (p < 0.05)

In [ ]:
# KPSS Test on Combined Differenced Series
kp = kpss(sddiff)
print("p-value for KPSS Test =", kp[1])
# Series is trend stationary (p > 0.05)

In [ ]:
plot_acf(sddiff)
plt.show()
plot_pacf(sddiff)
plt.show()
# We finally have confirmed that the series is now stationary, via plots as well as tests.